In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance


panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

In [2]:
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019, 2020, 2021])
).astype(int)

### Train/validate/test split
Train/validate/test split must be time-aware. Entries in the train set should predate entries in the validation set which should predate entries in the test set. The dataframe year_counts is used to decide which years should be used for the train/validation/test sets.

In [3]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


- 2022 and before used as train set
- 2023 used for validation set
- 2024 used for test set

In [4]:
panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
validate_X = validate.drop(columns = 'CompetesNextYear')
validate_y = validate['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']

### Establish baseline accuracy

In [5]:
clf_baseline = DummyClassifier(strategy="most_frequent")
clf_baseline.fit(train_X, train_y)
baseline_accuracy = clf_baseline.score(test_X, test_y)
print(baseline_accuracy)

0.5197880807636905


### Initial coarse feature selection using random forest feature importances

In [11]:
clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf.fit(train_X, train_y)

#will be using performance on validation set for feature selection
def evaluate_model(classifier, X_val, y_val):
    preds = classifier.predict(X_val)
    return {
        "f1":        f1_score(y_val, preds),
        "recall":    recall_score(y_val, preds),
        "accuracy":  accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds) 
    }

#performance of classifier in validation set when trained on all features. will be used for comparison.
all_feats_metrics = evaluate_model(clf, validate_X, validate_y)

Approach 1: permutation importance and then drop features with negative importance. 

<b>Approach 2 </b>: use random forest for feature importance.

In [ ]:
#using random forest to get feature importances 
clf_feat_selection = RandomForestClassifier(random_state=42)
clf_feat_selection.fit(train_X, train_y)
importances = clf_feat_selection.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)
feature_importance_df

In [ ]:
#getting rid of features of order of magnitude e-03 or lower and traing HistGradientBoostingClassifier
important_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.01]
features = important_feats['Feature'].to_list()
train_rf_X= train_X[features]
validate_rf_X = validate_X[features]

clf_rf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_rf.fit(train_rf_X, train_y)

rf_metrics = evaluate_model(clf_rf, validate_rf_X, validate_y)

In [20]:
result = permutation_importance(clf, validate_X, validate_y, n_repeats=10, random_state=42)

feature_importance_perm = pd.DataFrame({
    'Feature': validate_X.columns,
    'Importance': result.importances_mean
}).sort_values('Importance', ascending=False, ignore_index=True)

perm_feats = feature_importance_perm.loc[feature_importance_perm['Importance']>0, 'Feature'].to_list()
train_perm_X = train_X[perm_feats]
validate_perm_X = validate_X[perm_feats]
clf_perm_feats = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_perm_feats.fit(train_perm_X, train_y)

perm_metrics = evaluate_model(clf_perm_feats, validate_perm_X, validate_y)

In [21]:
print(all_feats_metrics)
print(rf_metrics)
print(perm_metrics)

{'f1': 0.6388302859821847, 'recall': 0.6278292921730115, 'accuracy': 0.6571062953793084, 'precision': 0.6502236802863107}
{'f1': 0.6433828835329773, 'recall': 0.6418821632206416, 'accuracy': 0.6562995521183965, 'precision': 0.6448906376576785}
{'f1': 0.6082134482540085, 'recall': 0.5527270632955135, 'accuracy': 0.6560491835201825, 'precision': 0.6760831278619233}


Will do a coarse feature reduction based on random forest feature ranking before training HistBoostClassifier, as this shows no meaningful change from using all features (tiny increase in f1 score and recall compared to using all features and negligible decrease in accuracy and precision). 

In [22]:
train_perc_X = train_rf_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])

validate_perc_X = validate_rf_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])


clf_perc = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_perc.fit(train_perc_X, train_y)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",6
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dtyp

In [24]:
perc_metrics = evaluate_model(clf_perc, validate_perc_X, validate_y)
print(rf_metrics)
print(perc_metrics)

{'f1': 0.6433828835329773, 'recall': 0.6418821632206416, 'accuracy': 0.6562995521183965, 'precision': 0.6448906376576785}
{'f1': 0.6388206388206388, 'recall': 0.628923573115245, 'accuracy': 0.6564942832503408, 'precision': 0.649034175334324}


Prioritising recall (not missing churners) on the assumption that the cost of a retention intervention is lower than the expected value of a lost customer. However, this should be verified. Therefore keeping the non-percentage based improvement features, as recall decreases when they are removed.

### Examining correlated features

In [ ]:
panel[['Wilks', 'Dots', 'Goodlift']].corr()

Wilks, Dots and Goodlift were all >0.99 correlated, so only Goodlift was retained. Goodlift is considered the most theoretically sound formula for normalising performance across weight classes, having been developed to address known biases in Wilks.

In [ ]:
train_gl_X = train_imp_X.drop(columns = ['Dots', 'Wilks'])

validate_gl_X = validate_imp_X.drop(columns = ['Dots', 'Wilks'])

clf_gl = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_gl.fit(train_gl_X, train_y)

In [ ]:
gl_metrics = evaluate_model(clf_gl, validate_gl_X, validate_y)
print(imp_metrics)
print(gl_metrics)

negligible performace change from dropping Wilks and Dots. 

### How many features to keep
Will reevaluate feature importance using random forest, in case dropping features has chnaged importance ranking.
Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.


In [ ]:
clf_ablation = RandomForestClassifier(random_state=42)
clf_ablation.fit(train_gl_X, train_y)
importances_ablation = clf_ablation.feature_importances_
ablation_df = pd.DataFrame({
    'Feature': train_gl_X.columns,
    'Importance': importances_ablation
}).sort_values('Importance', ascending=False, ignore_index =True)
ablation_df

# reduced_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.01]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0).reset_index(drop = True)



Can also see that percentage-based improvement features perform better than raw improvement. Will drop raw improvement features

Wilks, Goodlift and Dots are all different formulas for bodyweight adjusted strength. Very high correlation between these so will only keep most predictive feature, found to be Dots.

In [ ]:
# reduced_feats = feature_importance_df.loc[:13]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0)
# reduced_feats[:4]

In [ ]:
results = []

for i in range(len(feature_importance_df)):
    features = feature_importance_df.loc[:i, 'Feature'].to_list()
    columns = features + ['CompetesNextYear']

    train_n = train[columns]
    train_n_X = train_n.drop(columns = 'CompetesNextYear')
    train_n_y = train_n['CompetesNextYear']

    validate_n = validate[columns]
    validate_n_X = validate_n.drop(columns = 'CompetesNextYear')
    validate_n_y = validate_n['CompetesNextYear']

    clf_n = RandomForestClassifier(random_state=42)
    clf_n.fit(train_n_X, train_n_y)
    
    preds_n = clf_n.predict(validate_n_X)
    acc_n = accuracy_score(validate_n_y, preds_n)
    f1_n = f1_score(validate_n_y, preds_n)
    precision_n = precision_score(validate_n_y, preds_n)
    recall_n = recall_score(validate_n_y, preds_n)

    results.append({'n_features': len(features), 'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision_n': precision_n, 'recall_n':recall_n})

results_df = pd.DataFrame(results)

In [ ]:
results_df